In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import math
import pandas as pd
import numpy as np
import joblib
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [2]:
# Load the datasets
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\python scripts\\Catagorical prediction\\k-NN')
with open('accuracy.pkl', 'rb') as f:
    a = pickle.load(f)
with open('round.pkl', 'rb') as f:
    r = pickle.load(f)
    
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\Testing and Analysis\\2024 cleaned data')
match_results = pd.read_csv(f'2024 {r} afl_match_results_cleaned.csv')
team_stats = pd.read_csv(f'2024 {r} afl_team_stats_cleaned.csv')
win_streaks = pd.read_csv(f'2024 {r} afl_team_streaks_cleaned.csv',index_col=0)
venue_streaks = pd.read_csv(f'2024 {r} afl_venue_streaks_cleaned.csv',index_col=0)
team_form = pd.read_csv(f'2024 {r} afl_team_form_cleaned.csv',index_col=0)
fixture = pd.read_csv(f'2024 {r} afl_fixture_cleaned.csv',index_col=0)
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\python scripts\\Catagorical prediction\\k-NN')

In [3]:
def round_decimals_up(number:float, decimals:int=2):
    """
    Returns a value rounded up to a specific number of decimal places.
    """
    if not isinstance(decimals, int):
        raise TypeError("decimal places must be an integer")
    elif decimals < 0:
        raise ValueError("decimal places has to be 0 or more")
    elif decimals == 0:
        return math.ceil(number)

    factor = 10 ** decimals
    return math.ceil(number * factor) / factor

def extract_features(home_team, away_team, venue,weather):
    # Get the weighted average stats for home and away teams
    home_stats = team_stats[team_stats['Team'] == home_team].iloc[:, 1:].values.flatten()
    away_stats = team_stats[team_stats['Team'] == away_team].iloc[:, 1:].values.flatten()
    
    # Get the win streaks
    team_win_streak = win_streaks.loc[away_team, home_team].flatten()
    home_venue_streak = venue_streaks.loc[home_team, venue].flatten()
    away_venue_streak = venue_streaks.loc[away_team, venue].flatten()
    home_team_form = team_form.loc[team_form['Team'] == home_team, 'Current.Form'].values[0].flatten()
    away_team_form = team_form.loc[team_form['Team'] == home_team, 'Current.Form'].values[0].flatten()
    
    # Combine all features into a single array
    features = np.concatenate([
    home_stats, home_team_form,
    away_stats, away_team_form,
    home_venue_streak,away_venue_streak,team_win_streak,
    ])

    df1 = pd.DataFrame([features])
    df2 = pd.DataFrame([weather])
    features=pd.concat([df1, df2], axis = 1)

    column_names = match_results.drop(columns=['match.homeTeam.name', 'match.awayTeam.name','venue.name','Margin','Result','weather.weatherType',
                                          'Home.Team.Venue.Win.Streak', 'Away.Team.Venue.Win.Streak','Home.Win.Streak']).columns  # Replace with actual feature names
    column_names = column_names.append(pd.Index(['Home.Team.Venue.Win.Streak', 'Away.Team.Venue.Win.Streak','Home.Win.Streak'])).append(pd.Index(['weather.weatherType']))
    
    features.columns = column_names
    
    return features

def make_prediction(home_team, away_team, venue,weather):
    features = extract_features(home_team, away_team, venue,weather)
    pred_probs = model.predict_proba(features)  # Get the probability for each class
    pred_class = np.argmax(pred_probs, axis=1)  # Class with the highest probability
    pred_class = encoder.inverse_transform([pred_class])[0]
    predicted_prob = np.max(pred_probs, axis=1)
    acc = predicted_prob[0] * a
    max_prob_percent = f"{acc * 100:.2f}%"
    market = f"{round_decimals_up(1 / acc,2):.2f}"
    
    return pred_class,max_prob_percent,market

In [4]:
with open('encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)
with open('preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)
model = joblib.load('knn_model.pkl')

weather_categories = ['CLEAR_NIGHT','MOSTLY_SUNNY','OVERCAST','RAIN','SUNNY','THUNDERSTORMS','WINDY']  # Add all weather types you used

# Create a dictionary where all categories are 0
weather_dict = {category: 0 for category in weather_categories}

results=[]
prob=[]
market=[]
for home_team,away_team,venue,weather in zip(list(fixture['home.team.name']),
                                             list(fixture['away.team.name']),
                                             list(fixture['venue.name']),
                                             list(fixture['Next_round_weather'])):
    (r,p,m)=make_prediction(home_team,away_team,venue,weather)
    
    
    
    results.append(r)
    prob.append(p)
    market.append(m)

In [5]:
# # Run after all games
os.chdir('C:\\Users\\blake\\Desktop\\AFL Odds\\Testing and Analysis\\Catagorical prediction\\k-NN')
pd.DataFrame(results).to_excel(f"2024 k-NN testing {fixture['round.name'][1]}.xlsx", index=False)